# Corrigé — Jour 1 Matin : Fondamentaux LLM

## TP 1.1 — Installation de l'assistant IA

**Pas de corrigé déterministe** (ça dépend de votre OS et votre IDE), mais voici la **check-list de validation** :

- [ ] Ollama installé (`ollama --version` retourne une version)
- [ ] Modèle téléchargé (`ollama list` affiche `llama3.1:8b` ou similaire)
- [ ] Extension Continue installée dans VS Code
- [ ] Fichier `~/.continue/config.json` modifié
- [ ] Test : taper un commentaire `# fonction X` dans un fichier .py et accepter la complétion

### Si ça ne marche pas
- **Ollama ne démarre pas** : vérifier que le port 11434 n'est pas occupé.
- **Latence très haute** : votre machine est trop modeste, tester `phi3:mini` (plus léger).
- **Pas de complétion** : redémarrer VS Code après modification du config.json.

---

## TP 1.2 — Calculer la facture LLM

In [ ]:
# ========================================================================
# Corrige TP 1.2
# ========================================================================

PR_PAR_MOIS  = 1200
TOKENS_IN    = 8000
TOKENS_OUT   = 2500

SONNET_IN, SONNET_OUT, SONNET_CACHE = 3.0, 15.0, 0.30
HAIKU_IN, HAIKU_OUT = 0.80, 4.0

def cout_appel(t_in, t_out, prix_in, prix_out, t_cache=0, prix_cache=0):
    """Calcule le cout d'UN appel en USD."""
    # Tokens d'input qui ne viennent PAS du cache
    in_paye_plein = t_in - t_cache
    # Conversion en millions puis multiplication par le prix
    cout_in = (in_paye_plein / 1_000_000) * prix_in
    cout_out = (t_out / 1_000_000) * prix_out
    cout_c = (t_cache / 1_000_000) * prix_cache
    return cout_in + cout_out + cout_c

# --- Q1 : Sonnet sans cache
c_q1 = cout_appel(TOKENS_IN, TOKENS_OUT, SONNET_IN, SONNET_OUT) * PR_PAR_MOIS
print(f"Q1 Sonnet sans cache : {c_q1:.2f} USD/mois")

# --- Q2 : Sonnet avec 5000 tokens en cache
c_q2 = cout_appel(TOKENS_IN, TOKENS_OUT, SONNET_IN, SONNET_OUT,
                  t_cache=5000, prix_cache=SONNET_CACHE) * PR_PAR_MOIS
gain_pct = (c_q1 - c_q2) / c_q1 * 100
print(f"Q2 Sonnet avec cache : {c_q2:.2f} USD/mois  (gain {gain_pct:.1f} %)")

# --- Q3 : Haiku 4.5
c_q3 = cout_appel(TOKENS_IN, TOKENS_OUT, HAIKU_IN, HAIKU_OUT) * PR_PAR_MOIS
ratio = c_q1 / c_q3
print(f"Q3 Haiku 4.5         : {c_q3:.2f} USD/mois  (x{ratio:.1f} moins cher)")
print("   Risque qualite : tests moins idiomatiques. A valider sur 50 PR avant bascule.")

# --- Q4 : Sonnet avec max_tokens = 1500
c_q4 = cout_appel(TOKENS_IN, 1500, SONNET_IN, SONNET_OUT) * PR_PAR_MOIS
print(f"Q4 Sonnet max 1500   : {c_q4:.2f} USD/mois")
print("   Risque : tests tronques pour les PR larges - fallback a prevoir.")

### Lecture des résultats attendus

- **Q1** ≈ 333 USD/mois.
- **Q2** ≈ 318 USD/mois → gain ~4,5 % (modeste ici car le cache est faible par rapport au total input).
- **Q3** ≈ 19,7 USD/mois → **× 17 moins cher**, mais risque qualité réel.
- **Q4** ≈ 244 USD/mois — gain en réduisant la sortie, à condition d'accepter les troncatures.

### Q5 — Recommandation

Une approche progressive :
1. **Étape 1** : activer le cache sur Sonnet → gain immédiat sans risque (Q2).
2. **Étape 2** : router les PR triviales (renommage, lint, doc) vers Haiku (économie majeure).
3. **Étape 3** : mesurer la qualité réelle sur 50 PR aléatoires avant toute bascule définitive.
4. **Étape 4** : ajuster `max_tokens` cas par cas — pas un réglage global.
